# Exercice, Appliquer un extrait "update" KBO sur bronze et silver

Vous avez déjà un dossier d'extrait journalier téléchargé, par exemple
`KboOpenData_0432_2026_07_26_Update/`, qui contient :

- `meta.csv` : numéro d'extrait (`ExtractNumber`), type (`ExtractType`),
  date du snapshot (`SnapshotDate`)
- pour chaque entité (`enterprise`, `establishment`, `branch`,
  `denomination`, `address`, `contact`, `activity`) : un fichier
  `{entite}_insert.csv` et un fichier `{entite}_delete.csv`

**Important** pour`enterprise`/`establishment`/`branch`, chaque ligne d'insert a une clé
unique (`EnterpriseNumber`/`EstablishmentNumber`/`Id`)

Mais pour `denomination`/`address`/`contact`/`activity`, le fichier delete
ne liste QUE `EntityNumber` (pas de clé de ligne précise), ça veut dire
que dès qu'UN SEUL élément change pour une entité (ex: une seule activité
NACE ajoutée), TOUTES les lignes de cette entité dans cette table sont
supprimées puis réinsérées en entier.

vous allez :
1. Lire `meta.csv` et les fichiers insert/delete (assurez vous de pas appliquer la meme mise a jour deux fois)
2. Appliquer la mise à jour sur bronze
3. Ne reconstruire `entreprise`/`entreprise_silver` QUE pour les
   entreprises réellement affectées par ce lot (pas un rebuild complet)
4. Éviter de rejouer deux fois le même extrait (meta.csv contient un snapshot)

**Mise à jour** : plusieurs extraits successifs sont maintenant disponibles
(`0432`, `0433`, `0434`...), à appliquer **dans l'ordre chronologique** puisque
chacun est un delta par rapport à l'état laissé par le précédent. Les sections
1 à 4 ci-dessous définissent donc des **fonctions réutilisables** (une par
étape, paramétrées par le dossier d'extrait) plutôt que du code à exécuter
directement sur un seul dossier codé en dur ; la section 5, à la fin, boucle
sur tous les extraits disponibles, dans le bon ordre, et applique chacun via
ces fonctions (chaque extrait restant individuellement protégé par la même
vérification "déjà appliqué ?" qu'avant).


## 1. Lire `meta.csv` et les fichiers insert/delete


In [6]:
from __future__ import annotations

import os
from pathlib import Path
import csv

import pymongo

MONGO_URI = os.environ.get("MONGO_URI", "mongodb://localhost:27017")
MONGO_DB = os.environ.get("MONGO_DB", "kbo")
client = pymongo.MongoClient(MONGO_URI)
db = client[MONGO_DB]

# L'énoncé nomme les collections bronze "kbo_enterprise", "kbo_establishment", etc.
# Dans les notebooks précédents de ce projet, on les avait appelées enterprise,
# establishment, etc. (SANS préfixe kbo_). Vérifiez dans Compass quel nom
# correspond à votre base réelle, et ajustez ce dict si besoin -- tout le reste
# du notebook passe systématiquement par ce dict, jamais par un nom en dur.
COLLECTION_NAMES = {
    "enterprise": "enterprise",
    "establishment": "establishment",
    "branch": "branch",
    "denomination": "denomination",
    "address": "address",
    "contact": "contact",
    "activity": "activity",
}
ENTITIES = list(COLLECTION_NAMES)


def read_csv_rows(path: Path) -> list[dict]:
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def read_meta(update_dir: Path) -> dict:
    """meta.csv est au format Variable/Value (une ligne par variable), pas une
    table classique à une ligne par enregistrement."""
    return {row["Variable"]: row["Value"] for row in read_csv_rows(update_dir / "meta.csv")}


def load_extract(update_dir: Path) -> tuple[dict, dict, dict]:
    """Lit meta.csv + les 14 fichiers insert/delete d'un dossier d'extrait donné.
    Renvoie (meta, inserts, deletes)."""
    meta = read_meta(update_dir)
    inserts = {entity: read_csv_rows(update_dir / f"{entity}_insert.csv") for entity in ENTITIES}
    deletes = {entity: read_csv_rows(update_dir / f"{entity}_delete.csv") for entity in ENTITIES}
    return meta, inserts, deletes


def refresh_code_reference(update_dir: Path) -> None:
    """code.csv est fourni en PLUS des 7 entités du sujet : c'est une copie
    complète et à jour du référentiel de traduction (pas un delta insert/delete).
    On le recharge intégralement à chaque extrait -- petite table, sans coût."""
    code_rows = read_csv_rows(update_dir / "code.csv")
    if code_rows:
        db.code.delete_many({})
        db.code.insert_many(code_rows)


def already_applied(extract_number: int) -> bool:
    """Ne pas rejouer deux fois le même extrait (le log lui-même est écrit en
    section 4, une fois l'extrait appliqué avec succès)."""
    return db.kbo_update_log.find_one({"extractNumber": extract_number}) is not None


# Petite démonstration sur le premier extrait disponible, juste pour vérifier
# que la lecture fonctionne (l'application réelle, elle, se fait en section 5,
# sur TOUS les extraits, dans l'ordre) :
demo_dir = Path("../data/kbo_update_0432")
demo_meta, demo_inserts, demo_deletes = load_extract(demo_dir)
print("Exemple lu :", demo_meta)
for entity in ENTITIES:
    print(f"{entity:15s} insert={len(demo_inserts[entity]):4d}  delete={len(demo_deletes[entity]):4d}")


Exemple lu : {'SnapshotDate': '25-07-2026', 'ExtractTimestamp': '26-07-2026 09:53:01', 'ExtractType': 'update', 'ExtractNumber': '432', 'Version': '1.0.0'}
enterprise      insert=  16  delete= 139
establishment   insert=   8  delete=  13
branch          insert=   0  delete=   0
denomination    insert=  39  delete= 155
address         insert=  42  delete=  43
contact         insert=  28  delete=   9
activity        insert= 131  delete=  27


## 2. Appliquer la mise à jour sur bronze

Pour chacune des 7 entités, appliquez le insert/delete sur sa collection
brute (`kbo_enterprise`, `kbo_establishment`, `kbo_branch`,
`kbo_denomination`, `kbo_address`, `kbo_contact`, `kbo_activity`) :

In [8]:
from pymongo import ReplaceOne
from pymongo.errors import BulkWriteError


def resolve_enterprise_number(entity_number: str) -> str | None:
    """Retrouve l'EnterpriseNumber propriétaire d'un EntityNumber, qu'il désigne
    directement une entreprise, un établissement, ou une succursale (EntityNumber
    est un identifiant générique partagé par les 3 niveaux). À appeler AVANT
    toute mutation du bronze pour ce lot : une fois un établissement/succursale
    supprimé, son propriétaire n'est plus retrouvable."""
    if db[COLLECTION_NAMES["enterprise"]].find_one({"EnterpriseNumber": entity_number}, {"_id": 1}):
        return entity_number
    establishment = db[COLLECTION_NAMES["establishment"]].find_one(
        {"EstablishmentNumber": entity_number}, {"EnterpriseNumber": 1}
    )
    if establishment:
        return establishment["EnterpriseNumber"]
    branch = db[COLLECTION_NAMES["branch"]].find_one({"Id": entity_number}, {"EnterpriseNumber": 1})
    if branch:
        return branch["EnterpriseNumber"]
    return None  # EntityNumber inconnu du bronze actuel (rare, mais on ne casse pas pour ça)


def compute_affected_enterprise_numbers(inserts: dict, deletes: dict) -> set[str]:
    affected = set()

    for row in inserts["enterprise"] + deletes["enterprise"]:
        affected.add(row["EnterpriseNumber"])

    for row in inserts["establishment"]:
        affected.add(row["EnterpriseNumber"])
    for row in deletes["establishment"]:
        owner = resolve_enterprise_number(row["EstablishmentNumber"])
        if owner:
            affected.add(owner)

    for row in inserts["branch"]:
        affected.add(row["EnterpriseNumber"])
    for row in deletes["branch"]:
        owner = resolve_enterprise_number(row["Id"])
        if owner:
            affected.add(owner)

    for entity in ["denomination", "address", "contact", "activity"]:
        entity_numbers = {row["EntityNumber"] for row in inserts[entity] + deletes[entity]}
        for entity_number in entity_numbers:
            owner = resolve_enterprise_number(entity_number)
            if owner:
                affected.add(owner)

    return affected


def apply_row_key_update(entity: str, primary_key: str, inserts: dict, deletes: dict, extra_id_field: str | None = None) -> None:
    """enterprise/establishment/branch : chaque ligne a une clé propre, on peut
    donc faire un delete ciblé puis un upsert ligne par ligne."""
    collection = db[COLLECTION_NAMES[entity]]

    delete_keys = [row[primary_key] for row in deletes[entity]]
    if delete_keys:
        result = collection.delete_many({primary_key: {"$in": delete_keys}})
        print(f"{entity:15s} bronze : {result.deleted_count} supprimé(s)")

    if inserts[entity]:
        operations = []
        for row in inserts[entity]:
            doc = dict(row)
            if extra_id_field:
                doc["_id"] = doc[extra_id_field]  # _id = EnterpriseNumber, comme dans le TD1
            operations.append(ReplaceOne({primary_key: row[primary_key]}, doc, upsert=True))
        try:
            result = collection.bulk_write(operations, ordered=False)
            print(f"{entity:15s} bronze : {result.upserted_count} inséré(s), {result.modified_count} mis à jour")
        except BulkWriteError as exc:
            print(f"{entity:15s} bronze : erreurs bulk_write :", exc.details["writeErrors"])
            raise


def apply_detail_update(entity: str, inserts: dict, deletes: dict) -> None:
    """denomination/address/contact/activity : le delete ne connaît que
    EntityNumber (pas de clé de ligne) -> on supprime TOUTES les lignes de
    cette entité puis on réinsère l'intégralité des lignes fournies par insert."""
    collection = db[COLLECTION_NAMES[entity]]

    delete_entity_numbers = [row["EntityNumber"] for row in deletes[entity]]
    if delete_entity_numbers:
        result = collection.delete_many({"EntityNumber": {"$in": delete_entity_numbers}})
        print(f"{entity:15s} bronze : {result.deleted_count} ligne(s) supprimée(s) avant réinsertion")

    if inserts[entity]:
        result = collection.insert_many(inserts[entity])
        print(f"{entity:15s} bronze : {len(result.inserted_ids)} ligne(s) réinsérée(s)")


def apply_bronze_update(inserts: dict, deletes: dict) -> set[str]:
    """Calcule l'ensemble affecté (AVANT toute mutation), puis applique
    réellement l'insert/delete sur les 7 collections bronze. Renvoie l'ensemble
    affecté, à transmettre tel quel à propagate_update (section 3)."""
    affected_enterprise_numbers = compute_affected_enterprise_numbers(inserts, deletes)
    print(f"{len(affected_enterprise_numbers)} entreprise(s) affectée(s) par ce lot (calculé avant mise à jour du bronze)")

    apply_row_key_update("enterprise", "EnterpriseNumber", inserts, deletes, extra_id_field="EnterpriseNumber")
    apply_row_key_update("establishment", "EstablishmentNumber", inserts, deletes)
    apply_row_key_update("branch", "Id", inserts, deletes)
    for entity in ["denomination", "address", "contact", "activity"]:
        apply_detail_update(entity, inserts, deletes)

    return affected_enterprise_numbers


## 3. Propager vers `entreprise` et `entreprise_silver` !! SEULEMENT pour les entreprises affectées !!

Un rebuild complet de `entreprise_silver` (drop + réinsertion de toute la
base) serait du gâchis pour un lot qui ne touche que quelques centaines
d'entreprises sur des millions.

Attention à l'ORDRE : calculez l'ensemble affecté AVANT d'avoir appliqué les
deletes de l'étape précédente (sinon vous ne pourrez plus retrouver le
propriétaire d'un établissement/succursale déjà supprimé).


In [9]:
def _detail_lookup_stages(primary_key: str) -> list[dict]:
    joins = [
        (COLLECTION_NAMES["denomination"], "denominations"),
        (COLLECTION_NAMES["address"], "addresses"),
        (COLLECTION_NAMES["contact"], "contacts"),
        (COLLECTION_NAMES["activity"], "activities"),
    ]
    return [
        {"$lookup": {"from": collection, "localField": primary_key, "foreignField": "EntityNumber", "as": alias}}
        for collection, alias in joins
    ]


def _establishments_lookup_stage() -> dict:
    return {
        "$lookup": {
            "from": COLLECTION_NAMES["establishment"],
            "let": {"enterpriseNumber": "$EnterpriseNumber"},
            "pipeline": [
                {"$match": {"$expr": {"$eq": ["$EnterpriseNumber", "$$enterpriseNumber"]}}},
                *_detail_lookup_stages("EstablishmentNumber"),
            ],
            "as": "establishments",
        }
    }


def _branches_lookup_stage() -> dict:
    return {
        "$lookup": {
            "from": COLLECTION_NAMES["branch"],
            "let": {"enterpriseNumber": "$EnterpriseNumber"},
            "pipeline": [
                {"$match": {"$expr": {"$eq": ["$EnterpriseNumber", "$$enterpriseNumber"]}}},
                *_detail_lookup_stages("Id"),
            ],
            "as": "branches",
        }
    }


def rebuild_entreprise_bronze(enterprise_numbers: set[str]) -> None:
    """Même pipeline de $lookup que le TD1, limité à `enterprise_numbers`, et
    écrit via $merge (upsert ciblé) plutôt que $out (qui viderait toute la
    collection)."""
    if not enterprise_numbers:
        return
    pipeline = [
        {"$match": {"EnterpriseNumber": {"$in": list(enterprise_numbers)}}},
        *_detail_lookup_stages("EnterpriseNumber"),
        _establishments_lookup_stage(),
        _branches_lookup_stage(),
        {"$merge": {"into": "entreprise", "whenMatched": "replace", "whenNotMatched": "insert"}},
    ]
    db[COLLECTION_NAMES["enterprise"]].aggregate(pipeline)


def split_still_existing_vs_removed(affected_enterprise_numbers: set[str]) -> tuple[set[str], set[str]]:
    """À appeler APRÈS apply_bronze_update : regarde, dans le bronze déjà mis à
    jour, lesquelles des entreprises affectées existent encore (à reconstruire)
    et lesquelles ont réellement disparu (à nettoyer)."""
    still_existing = {
        doc["_id"]
        for doc in db[COLLECTION_NAMES["enterprise"]].find(
            {"EnterpriseNumber": {"$in": list(affected_enterprise_numbers)}},
            {"_id": 1},
        )
    }
    removed = affected_enterprise_numbers - still_existing
    return still_existing, removed


In [14]:
code_map: dict = {}  # rechargé à chaque appel de build_silver_document via refresh_code_map()


def refresh_code_map() -> None:
    global code_map
    code_map = {(doc["Category"], doc["Code"]): doc["Description"] for doc in db.code.find({"Language": "FR"})}


def translate(category: str, raw_code: str):
    if not raw_code:
        return None
    return code_map.get((category, raw_code), raw_code)


SCALAR_CODE_FIELDS = {
    "Status": "status",
    "JuridicalSituation": "juridicalSituation",
    "TypeOfEnterprise": "typeOfEnterprise",
    "JuridicalForm": "juridicalForm",
    "JuridicalFormCAC": "juridicalFormCAC",
}


def build_silver_header(doc: dict) -> dict:
    header = {"_id": doc["_id"], "enterpriseNumber": doc.get("EnterpriseNumber", doc["_id"])}
    if doc.get("StartDate"):
        header["startDate"] = doc["StartDate"]
    for bronze_field, silver_field in SCALAR_CODE_FIELDS.items():
        raw = doc.get(bronze_field, "")
        if raw:
            category = "JuridicalForm" if bronze_field == "JuridicalFormCAC" else bronze_field
            header[silver_field] = translate(category, raw)
    return header


def build_denominations(entity_number: str) -> dict:
    out = {}
    for row in db[COLLECTION_NAMES["denomination"]].find({"EntityNumber": entity_number}):
        out[translate("TypeOfDenomination", row["TypeOfDenomination"])] = {
            "language": translate("Language", row["Language"]),
            "denomination": row["Denomination"],
        }
    return out


def clean_country(raw_country: str) -> str:
    import re
    cleaned = re.sub(r"\([^)]*\)", "", raw_country)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned or "Belgique"


def build_addresses(entity_number: str) -> dict:
    out = {}
    for row in db[COLLECTION_NAMES["address"]].find({"EntityNumber": entity_number}):
        address = {"country": clean_country(row.get("CountryFR", ""))}
        for bronze_field, silver_field in [
            ("Zipcode", "zipcode"), ("MunicipalityFR", "municipality"),
            ("StreetFR", "street"), ("HouseNumber", "houseNumber"), ("Box", "box"),
        ]:
            value = row.get(bronze_field, "")
            if value:
                address[silver_field] = value
        out[translate("TypeOfAddress", row["TypeOfAddress"])] = address
    return out


CONTACT_TYPE_TO_FIELD = {"EMAIL": "email", "TEL": "phone", "WEB": "web", "FAX": "fax"}


def build_contacts(entity_number: str) -> dict:
    out = {}
    for row in db[COLLECTION_NAMES["contact"]].find({"EntityNumber": entity_number}):
        value = row.get("Value", "")
        if value:
            field = CONTACT_TYPE_TO_FIELD.get(row["ContactType"], row["ContactType"].lower())
            out[field] = value
    return out


def build_activities(entity_number: str) -> dict:
    best = {}
    for row in db[COLLECTION_NAMES["activity"]].find({"EntityNumber": entity_number}):
        nace_version = row["NaceVersion"]
        description = translate(f"Nace{nace_version}", row["NaceCode"])
        activity_group = translate("ActivityGroup", row["ActivityGroup"])
        key = (activity_group, description)
        if key not in best or int(nace_version) > int(best[key]["naceVersion"]):
            best[key] = {
                "activityGroup": activity_group, "description": description,
                "naceVersion": nace_version, "classification": row["Classification"],
            }
    result = {"main": [], "secondary": []}
    for activity in best.values():
        bucket = "main" if activity["classification"] == "MAIN" else "secondary"
        result[bucket].append({k: activity[k] for k in ("activityGroup", "description", "naceVersion")})
    return result


DETAIL_BUILDERS = {
    "denominations": build_denominations, "addresses": build_addresses,
    "contacts": build_contacts, "activities": build_activities,
}


def build_details(entity_number: str, include=("denominations", "addresses", "contacts", "activities")) -> dict:
    return {key: DETAIL_BUILDERS[key](entity_number) for key in include}


def build_establishments(enterprise_number: str) -> dict:
    out = {}
    for establishment in db[COLLECTION_NAMES["establishment"]].find({"EnterpriseNumber": enterprise_number}):
        entry = {}
        if establishment.get("StartDate"):
            entry["startDate"] = establishment["StartDate"]
        entry.update(build_details(establishment["EstablishmentNumber"]))
        out[establishment["EstablishmentNumber"]] = entry
    return out


def build_branches(enterprise_number: str) -> dict:
    out = {}
    for branch in db[COLLECTION_NAMES["branch"]].find({"EnterpriseNumber": enterprise_number}):
        entry = {}
        if branch.get("StartDate"):
            entry["startDate"] = branch["StartDate"]
        entry.update(build_details(branch["Id"], include=("addresses", "contacts")))
        out[branch["Id"]] = entry
    return out


def build_silver_document(enterprise_doc: dict) -> dict:
    enterprise_number = enterprise_doc["_id"]
    silver_doc = build_silver_header(enterprise_doc)
    silver_doc.update(build_details(enterprise_number))
    silver_doc["establishments"] = build_establishments(enterprise_number)
    silver_doc["branches"] = build_branches(enterprise_number)
    return silver_doc


def propagate_update(affected_enterprise_numbers: set[str]) -> None:
    """Fonction "chef d'orchestre" de la section 3 : à appeler juste après
    apply_bronze_update, avec l'ensemble qu'elle a renvoyé."""
    still_existing, removed = split_still_existing_vs_removed(affected_enterprise_numbers)

    if removed:
        db.entreprise.delete_many({"_id": {"$in": list(removed)}})
        db.entreprise_silver.delete_many({"_id": {"$in": list(removed)}})
        print(f"{len(removed)} entreprise(s) supprimée(s) de entreprise / entreprise_silver")

    print(f"{len(still_existing)} entreprise(s) à reconstruire dans entreprise / entreprise_silver")

    rebuild_entreprise_bronze(still_existing)

    refresh_code_map()  # le référentiel a pu être rechargé par refresh_code_reference() entre-temps
    rebuilt = 0
    for enterprise_doc in db[COLLECTION_NAMES["enterprise"]].find({"_id": {"$in": list(still_existing)}}):
        silver_doc = build_silver_document(enterprise_doc)
        db.entreprise_silver.replace_one({"_id": silver_doc["_id"]}, silver_doc, upsert=True)
        rebuilt += 1

    print(f"entreprise : {len(still_existing)} document(s) reconstruit(s) — entreprise_silver : {rebuilt} document(s) reconstruit(s)")


## 5. Appliquer plusieurs extraits successifs, dans l'ordre

Trois extraits sont maintenant disponibles (`0432`, `0433`, `0434`). Chacun est
un delta par rapport à l'état laissé par le précédent : il faut donc les
appliquer **dans l'ordre chronologique** des numéros d'extrait, jamais dans un
ordre arbitraire (l'ordre des dossiers sur le disque, l'ordre alphabétique des
noms de fichiers, etc. ne sont pas fiables).

Chaque extrait reste individuellement protégé par `already_applied()` : on peut
rejouer cette cellule autant de fois qu'on veut (par exemple après avoir
ajouté un nouvel extrait au dossier), les extraits déjà appliqués seront
simplement sautés.


In [16]:
UPDATE_DIRS = [
    Path("../data/kbo_update_0432"),
    Path("../data/kbo_update_0433"),
    Path("../data/kbo_update_0434"),
]


def apply_update_extract(update_dir: Path) -> None:
    """Applique un seul dossier d'extrait, de bout en bout (sections 1 à 4).
    Ne fait rien si cet extrait est déjà dans kbo_update_log."""
    meta, inserts, deletes = load_extract(update_dir)
    extract_number = int(meta["ExtractNumber"])

    if already_applied(extract_number):
        print(f"Extrait #{extract_number} déjà appliqué, on saute.\n")
        return

    print(f"--- Application de l'extrait #{extract_number} (snapshot {meta['SnapshotDate']}) ---")
    refresh_code_reference(update_dir)
    affected = apply_bronze_update(inserts, deletes)
    propagate_update(affected)
    log_extract_applied(meta)
    print(f"Extrait #{extract_number} appliqué avec succès.\n")


# Tri par numéro d'extrait AVANT application : c'est ce qui garantit l'ordre
# chronologique, indépendamment de l'ordre de la liste ci-dessus.
for update_dir in sorted(UPDATE_DIRS, key=lambda d: int(read_meta(d)["ExtractNumber"])):
    apply_update_extract(update_dir)


--- Application de l'extrait #432 (snapshot 25-07-2026) ---
211 entreprise(s) affectée(s) par ce lot (calculé avant mise à jour du bronze)
enterprise      bronze : 139 supprimé(s)
enterprise      bronze : 16 inséré(s), 0 mis à jour
establishment   bronze : 13 supprimé(s)
establishment   bronze : 8 inséré(s), 0 mis à jour
denomination    bronze : 156 ligne(s) supprimée(s) avant réinsertion
denomination    bronze : 39 ligne(s) réinsérée(s)
address         bronze : 43 ligne(s) supprimée(s) avant réinsertion
address         bronze : 42 ligne(s) réinsérée(s)
contact         bronze : 14 ligne(s) supprimée(s) avant réinsertion
contact         bronze : 28 ligne(s) réinsérée(s)
activity        bronze : 233 ligne(s) supprimée(s) avant réinsertion
activity        bronze : 131 ligne(s) réinsérée(s)
136 entreprise(s) supprimée(s) de entreprise / entreprise_silver
75 entreprise(s) à reconstruire dans entreprise / entreprise_silver
entreprise : 75 document(s) reconstruit(s) — entreprise_silver : 75 d